# 📘 Notebook 06: NoSQL

### Course: *From One Table to a Cluster: Working with Data at Scale*
**Instructor:** Ioannis Tsioulis

*Module B: Beyond Tables · Notebook 2 of 2 in this module · (6 of 12 overall)*

---

## 🎯 Learning objectives

By the end of this notebook you will be able to:

- Recognise data that does not fit comfortably in tables, and describe the **document** model
- Build a small **document store** by hand, with queries on nested fields
- Explain what is given up when the database no longer enforces a **schema**
- Decide between **embedding** related data and **referencing** it, with measurements for both
- Spread data over several machines by **sharding**, and implement **consistent hashing**
- Explain **replication**, **eventual consistency** and the **CAP theorem**, with a simulation
- Read and write real **MongoDB** queries

> **Prerequisites:** Notebooks 01, 02 and 04. Python dictionaries, lists and classes.
>
> 🔭 **Why this notebook matters:** for thirty years "database" meant "relational database". Around 2005 the largest web companies found that their data had outgrown a single machine, and that the guarantees of Module A (rigid tables, joins, transactions) were hard to keep once data was spread over hundreds of servers. They built systems that gave some of those guarantees up on purpose. The name **NoSQL** stuck to all of them. Understanding these systems means understanding exactly what was traded for what, so that you can tell when the trade is a good one and when it is a fashionable mistake.

> 📦 **Libraries used in this notebook.** None that need installing for the main part: we build the document store, the sharding and the replication ourselves, in plain Python, and compare with **SQLite** (Notebook 01).
>
> Section 8 uses **mongomock**, a small library that imitates the programming interface of **MongoDB**, the most widely used document database, entirely inside the notebook. With it you write real MongoDB queries without needing a database server or an account. Its installation line is in the cell that uses it.

In [ ]:
import bisect
import copy
import datetime
import hashlib
import json
import random
import sqlite3
import time

## 1. Data that does not want to be a table

### Intuition first
Our bookshop grows, and begins to sell more than books: e-readers, gift cards, reading lamps. Each kind of product has its own properties. A book has an author and a number of pages. An e-reader has a screen size and a battery life. A gift card has a value and nothing else.

In a relational database every row of a table has the same columns. We could give the `products` table a column for every property of every kind of product, and leave most of them `NULL` in every row. Or we could create a separate table for each kind of product, and join them. Neither is pleasant, and both must be redone whenever a new kind of product arrives.

There is a second awkwardness. A product has **reviews**: a list, of varying length, of small records. In a table design that means another table and another join, although nobody ever looks at a review without its product.

Look at how naturally the same information is written as nested Python data.

In [ ]:
catalogue = [
    {"name": "The Salt Road", "kind": "book", "price": 12.50,
     "details": {"author": "M. Ortega", "pages": 320, "language": "English"},
     "tags": ["fiction", "sea", "bestseller"],
     "reviews": [{"stars": 5, "text": "Could not put it down"}, {"stars": 4, "text": "A slow start"}]},

    {"name": "A Winter Atlas", "kind": "book", "price": 18.00,
     "details": {"author": "L. Brandt", "pages": 210, "language": "English"},
     "tags": ["travel", "maps"],
     "reviews": []},

    {"name": "PaperLight 7", "kind": "e-reader", "price": 129.00,
     "details": {"screen_inches": 7, "battery_weeks": 6, "waterproof": True},
     "tags": ["electronics", "bestseller"],
     "reviews": [{"stars": 3, "text": "Good screen, slow menus"}]},

    {"name": "Gift card", "kind": "gift card", "price": 25.00,
     "details": {"valid_months": 24},
     "tags": []},
]

print(json.dumps(catalogue[2], indent=2))

### Formal definition
A **document** is a self-contained record made of named fields, whose values may be numbers, text, lists, or other documents nested inside. It is usually written in **JSON**, the notation printed above, which maps directly onto Python dictionaries and lists.

A **document database** stores documents in **collections**. Unlike the rows of a table, the documents of a collection **need not have the same fields**. The structure travels with each document and is not declared in advance.

## 2. A document store by hand

Storing documents is trivial: a dictionary from an identifier to the document. The interesting part is **querying** them. We need to say things like *"the books under 15 euros"* or *"products whose details say they are waterproof"*, when the field we ask about may be nested, or missing altogether.

We will use the query notation of MongoDB, in which **a query is itself a document** describing what the result must look like:

| Query | Meaning |
|---|---|
| `{"kind": "book"}` | the field `kind` equals `"book"` |
| `{"price": {"$lt": 15}}` | `price` is less than 15 |
| `{"details.pages": {"$gte": 300}}` | the nested field `details.pages` is at least 300 |
| `{"tags": "bestseller"}` | the list `tags` contains `"bestseller"` |
| `{"details.author": {"$exists": True}}` | the document has such a field at all |

In [ ]:
MISSING = object()                       # a marker for "this document has no such field"

def get_path(document, path):
    """Follow a dotted path such as 'details.pages' into a nested document."""
    value = document
    for part in path.split("."):
        if isinstance(value, dict) and part in value:
            value = value[part]
        else:
            return MISSING
    return value

COMPARISONS = {
    "$gt":  lambda value, operand: value > operand,
    "$gte": lambda value, operand: value >= operand,
    "$lt":  lambda value, operand: value < operand,
    "$lte": lambda value, operand: value <= operand,
    "$ne":  lambda value, operand: value != operand,
    "$in":  lambda value, operand: value in operand,
}

def matches(document, query):
    for path, condition in query.items():
        value = get_path(document, path)
        if isinstance(condition, dict):                          # operators such as {"$lt": 15}
            for operator, operand in condition.items():
                if operator == "$exists":
                    if (value is not MISSING) != operand:
                        return False
                elif value is MISSING:
                    return False
                else:
                    try:
                        if not COMPARISONS[operator](value, operand):
                            return False
                    except TypeError:                            # a text compared with a number: no match
                        return False
        elif value is MISSING:
            return False
        elif isinstance(value, list):                            # a list matches if it contains the value
            if condition not in value:
                return False
        elif value != condition:
            return False
    return True

class Collection:
    def __init__(self):
        self.documents = {}
        self.next_id = 1

    def insert_one(self, document):
        document = copy.deepcopy(document)
        document["_id"] = self.next_id
        self.documents[self.next_id] = document
        self.next_id += 1
        return document["_id"]

    def find(self, query=None):
        return [document for document in self.documents.values() if matches(document, query or {})]

    def find_one(self, query):
        if set(query) == {"_id"}:                                # by identifier: go straight to it
            return self.documents.get(query["_id"])
        found = self.find(query)
        return found[0] if found else None

    def update_one(self, query, change):
        document = self.find_one(query)
        if document is None:
            return 0
        for field, value in change.get("$set", {}).items():
            document[field] = value
        for field, amount in change.get("$inc", {}).items():
            document[field] = document.get(field, 0) + amount
        for field, item in change.get("$push", {}).items():
            document.setdefault(field, []).append(item)
        return 1

products = Collection()
for document in catalogue:
    products.insert_one(document)

def names(documents):
    return [document["name"] for document in documents]

print("Books:                  ", names(products.find({"kind": "book"})))
print("Under 20 euros:         ", names(products.find({"price": {"$lt": 20}})))
print("At least 300 pages:     ", names(products.find({"details.pages": {"$gte": 300}})))
print("Tagged bestseller:      ", names(products.find({"tags": "bestseller"})))
print("Waterproof:             ", names(products.find({"details.waterproof": True})))
print("Books or gift cards < 20:", names(products.find({"kind": {"$in": ["book", "gift card"]}, "price": {"$lt": 20}})))

About sixty lines, and we can query records of four different shapes with one notation. A document that lacks the field asked about is simply not a match: the gift card has no `details.pages`, and nobody had to store a `NULL` to say so.

Changes work on documents too. A new review is **pushed** onto the list inside the product, with no second table involved.

In [ ]:
products.update_one({"name": "A Winter Atlas"}, {"$push": {"reviews": {"stars": 5, "text": "Beautiful maps"}},
                                                 "$inc": {"price": -2.0}})

atlas = products.find_one({"name": "A Winter Atlas"})
print(atlas["price"], atlas["reviews"])

## 3. The price of freedom: no schema

That flexibility was bought with something. In Notebook 01 the database refused a customer without a name and an order for a customer who did not exist. A document store accepts **anything**.

In [ ]:
products.insert_one({"name": "Paper Lanterns", "kind": "book", "prcie": 9.90,           # a typing mistake in the field name
                     "details": {"author": "S. Okafor", "pages": "184"}})                # the page count as text

products.insert_one({"name": "Reading lamp", "kind": "Book", "price": "14.50"})         # wrong kind, price as text

print("Books:           ", names(products.find({"kind": "book"})))
print("Under 20 euros:  ", names(products.find({"price": {"$lt": 20}})))

Both insertions succeeded without a murmur, and the damage is silent.

- *Paper Lanterns* costs 9.90 and does not appear among the products under 20 euros. Its price was stored under the name `prcie`, so as far as any query is concerned it has no price.
- The *Reading lamp* is filed as a `Book` with a capital letter, and matches neither `book` nor anything else you would think of searching for. Its price of 14.50 was stored as the text `"14.50"`, and a text is never "less than 20", so it is missing from the second list too.

Nothing warned us, and nothing will, until a customer or an accountant notices.

### Formal definition
A relational database applies **schema-on-write**: the structure is declared once, and every write is checked against it. A document store applies **schema-on-read**: anything may be written, and making sense of it is the job of whoever reads it.

The schema has not disappeared. It has moved out of the database and into the code of every program that reads the data. The same rules must now be written, and kept consistent, in each of them. A minimal defence is to check documents ourselves before storing them.

In [ ]:
def problems_with(document):
    problems = []
    for field, expected_type in [("name", str), ("kind", str), ("price", (int, float))]:
        if field not in document:
            problems.append(f"missing field '{field}'")
        elif not isinstance(document[field], expected_type):
            problems.append(f"'{field}' has the wrong type: {type(document[field]).__name__}")
    if document.get("kind") not in ("book", "e-reader", "gift card", "lamp"):
        problems.append(f"unknown kind: {document.get('kind')!r}")
    return problems

for document in products.find():
    found = problems_with(document)
    print(f"{document['name']:<16} {'ok' if not found else found}")

> ⚠️ **Common pitfalls**
>
> - Believing that "schemaless" means "no need to design". The design decisions are all still there, and now nothing enforces them.
> - Letting field names and types drift between documents written at different times by different versions of a program. After two years a collection can contain five dialects of the same record.
> - Choosing a document database because the data looked flexible on the first day. Most business data (customers, orders, payments) is highly regular, and benefits from being checked.

## 4. Embed or reference?

The central design question of a document database has no counterpart in Module A. When two things are related, do you put one **inside** the other, or store them separately and keep a reference?

For orders, the document approach is to **embed**: an order document contains its lines, and each line contains the title, category and price of the book, and the order contains the customer's name and city. Everything about the order is in one place.

Let us build the same shop both ways: as four relational tables, and as one collection of order documents.

In [ ]:
def make_shop(customers=1000, products=200, orders=10000, seed=0):
    rng = random.Random(seed)
    cities = ["Athens", "Thessaloniki", "Patras", "Heraklion", "Larissa", "Volos", "Ioannina", "Chania"]
    categories = ["Fiction", "Science", "History", "Children", "Cooking", "Travel", "Art", "Computing"]
    customer_rows = [(i, f"Customer {i}", rng.choice(cities), 1950 + rng.randint(0, 55)) for i in range(1, customers + 1)]
    product_rows = [(i, f"Book {i}", rng.choice(categories), round(rng.uniform(5, 60), 2)) for i in range(1, products + 1)]
    order_rows, item_rows = [], []
    first_day = datetime.date(2023, 1, 1)
    for order_id in range(1, orders + 1):
        day = first_day + datetime.timedelta(days=rng.randint(0, 729))
        order_rows.append((order_id, rng.randint(1, customers), day.isoformat()))
        for line in range(rng.randint(1, 4)):
            item_rows.append((len(item_rows) + 1, order_id, rng.randint(1, products), rng.randint(1, 3)))
    return customer_rows, product_rows, order_rows, item_rows

customer_rows, product_rows, order_rows, item_rows = make_shop(orders=20000)

# --- the relational version, with the indexes of Notebook 03
db = sqlite3.connect(":memory:")
db.executescript("""
    CREATE TABLE customers   (customer_id INTEGER PRIMARY KEY, name TEXT, city TEXT, birth_year INTEGER);
    CREATE TABLE products    (product_id INTEGER PRIMARY KEY, title TEXT, category TEXT, price REAL);
    CREATE TABLE orders      (order_id INTEGER PRIMARY KEY, customer_id INTEGER, order_date TEXT);
    CREATE TABLE order_items (item_id INTEGER PRIMARY KEY, order_id INTEGER, product_id INTEGER, quantity INTEGER);
    CREATE INDEX items_by_order ON order_items(order_id);
""")
db.executemany("INSERT INTO customers VALUES (?, ?, ?, ?)", customer_rows)
db.executemany("INSERT INTO products VALUES (?, ?, ?, ?)", product_rows)
db.executemany("INSERT INTO orders VALUES (?, ?, ?)", order_rows)
db.executemany("INSERT INTO order_items VALUES (?, ?, ?, ?)", item_rows)
db.commit()

# --- the document version: one document per order, everything embedded
customer_by_id = {row[0]: row for row in customer_rows}
product_by_id = {row[0]: row for row in product_rows}
lines_by_order = {}
for item_id, order_id, product_id, quantity in item_rows:
    title, category, price = product_by_id[product_id][1:]
    lines_by_order.setdefault(order_id, []).append({"title": title, "category": category, "price": price, "quantity": quantity})

orders = Collection()
for order_id, customer_id, order_date in order_rows:
    name, city = customer_by_id[customer_id][1:3]
    orders.insert_one({"date": order_date, "customer": {"id": customer_id, "name": name, "city": city},
                       "lines": lines_by_order[order_id]})

print(json.dumps(orders.find_one({"_id": 7}), indent=2))

### What embedding gains
The commonest request in a shop is *"show me this order"*: its date, the customer, and every line with its title and price.

In [ ]:
random.seed(0)
wanted = [random.randint(1, len(order_rows)) for i in range(2000)]

whole_order = """
    SELECT o.order_date, c.name, c.city, p.title, p.category, p.price, i.quantity
    FROM orders AS o
    JOIN customers AS c    ON c.customer_id = o.customer_id
    JOIN order_items AS i  ON i.order_id = o.order_id
    JOIN products AS p     ON p.product_id = i.product_id
    WHERE o.order_id = ?
"""
started = time.perf_counter()
for order_id in wanted:
    db.execute(whole_order, (order_id,)).fetchall()
relational_time = (time.perf_counter() - started) / len(wanted) * 1000

started = time.perf_counter()
for order_id in wanted:
    orders.find_one({"_id": order_id})
document_time = (time.perf_counter() - started) / len(wanted) * 1000

print(f"Relational: four tables joined      {relational_time:.4f} ms per order")
print(f"Document:   one lookup by identifier {document_time:.4f} ms per order")

The document arrives whole, from one place, with a single lookup. The relational database must visit four tables and assemble the answer. On one machine both are quick. Spread the data over a hundred machines, and the difference becomes decisive: one lookup goes to one machine, while a four-way join may have to gather its pieces from several.

### What embedding costs
A book was filed under the wrong category, and the mistake must be corrected. How many records change?

In [ ]:
book = "Book 17"

relational_changes = db.execute("SELECT COUNT(*) FROM products WHERE title = ?", (book,)).fetchone()[0]
document_changes = sum(1 for order in orders.find() for line in order["lines"] if line["title"] == book)

relational_size = db.execute("PRAGMA page_count").fetchone()[0] * db.execute("PRAGMA page_size").fetchone()[0]
document_size = sum(len(json.dumps(order)) for order in orders.find())

print(f"{'':<12} {'records to change':>18} {'size of the data':>18}")
print(f"{'Relational':<12} {relational_changes:>18,} {relational_size / 1e6:>15.1f} MB")
print(f"{'Documents':<12} {document_changes:>18,} {document_size / 1e6:>15.1f} MB")

One row in the relational design. Hundreds of order lines, scattered through thousands of documents, in the embedded one. Miss some, and the data contradicts itself. This is the **update anomaly** of Notebook 01 once more. Embedding is denormalisation, and it has the price that denormalisation always has.

| | Read one complete order | Correct a fact shared by many orders |
|---|---|---|
| Relational (referenced) | a join across four tables | one row |
| Documents (embedded) | one lookup | every document that contains a copy |
| Better choice | **documents** | **relational** |

The practical rule follows from the table. **Embed** what is always read together with its parent and belongs to it alone: the lines of an order, the reviews of a product. **Reference** what is shared and may change: the details of a customer or a product that appear in thousands of orders. And note one case where copying is exactly right: the price on an order line **should** be the price at the moment of sale, never to be updated, as with the facts of Notebook 04.

## 5. Key-value stores

Strip the document database down to its barest form. There are no queries on fields, no nesting that the database understands. There are only two operations: **put** a value under a key, and **get** it back. This is a **key-value store**, and in Python it is one line.

In [ ]:
store = {}

store["session:8f3a"] = {"customer": 618, "basket": ["Book 12", "Book 40"]}       # put
store["page:/books/12"] = "<html>...the rendered page...</html>"

print(store.get("session:8f3a"))                                                   # get
print(store.get("session:0000"))                                                   # a key that is not there

It looks too simple to be useful, and it is one of the most heavily used kinds of database there is. When all you need is *"give me the thing with this key"*, a key-value store can answer in well under a millisecond and hold billions of entries: shopping baskets, login sessions, cached pages, user settings.

Its limitation is the other side of its simplicity. You cannot ask *"which sessions have Book 12 in the basket?"* without fetching every value and looking inside. The database sees the values as opaque parcels.

The simplicity has one more consequence, and it is the reason this family of systems exists at all. If every operation concerns **one key**, then different keys have nothing to do with each other, and can be put on **different machines**.

## 6. Scaling out: sharding

### Intuition first
One machine is full. We buy three more, and must now decide which machine holds which key. Every program that reads or writes must be able to work out, from the key alone, where to go. Splitting data across machines like this is called **sharding**, and each machine's portion is a **shard**.

The obvious rule: turn the key into a number with a **hash function**, and take the remainder after dividing by the number of machines.

In [ ]:
def hash_number(key):
    return int(hashlib.md5(str(key).encode()).hexdigest(), 16)      # the same on every machine, unlike Python's hash()

def shard_by_remainder(key, machines):
    return hash_number(key) % machines

keys = [f"customer:{i}" for i in range(100000)]

load = [0, 0, 0, 0]
for key in keys:
    load[shard_by_remainder(key, 4)] += 1
print("Keys on each of 4 machines:", load)

The keys are spread almost perfectly evenly, and any program can compute where a key lives. So far, so good.

Now the shop grows again, and we add a **fifth** machine. How many keys have to move?

In [ ]:
moved = sum(1 for key in keys if shard_by_remainder(key, 4) != shard_by_remainder(key, 5))
print(f"Going from 4 machines to 5, {moved:,} of {len(keys):,} keys change machine: {moved / len(keys):.0%}")

Four fifths of all the data must be copied from one machine to another, for the sake of adding a fifth of extra capacity. During that time the system is slow or unavailable. With remainders, changing the number of machines reshuffles nearly everything.

### Consistent hashing
A better scheme was published in 1997, and is used by almost every distributed store built since.

Imagine the hash values arranged around a **circle**, like the hours on a clock face. Place each **machine** at a point on the circle, by hashing its name. To find where a key lives, hash the key to a point on the circle and walk **clockwise** to the first machine you meet.

When a machine is added, it takes its place on the circle and takes over only the keys that lie between it and the machine before it. Every other key stays exactly where it was.

In [ ]:
class Ring:
    def __init__(self, machines, points_per_machine=100):
        self.points = []                                     # (position on the circle, machine)
        self.points_per_machine = points_per_machine
        for machine in machines:
            self.add(machine)

    def add(self, machine):
        for i in range(self.points_per_machine):             # each machine sits at many points, for an even spread
            self.points.append((hash_number(f"{machine}#{i}"), machine))
        self.points.sort()

    def machine_for(self, key):
        position = bisect.bisect(self.points, (hash_number(key), ""))      # the first point clockwise from the key
        return self.points[position % len(self.points)][1]

ring = Ring(["machine-1", "machine-2", "machine-3", "machine-4"])
before = {key: ring.machine_for(key) for key in keys}

load = {}
for machine in before.values():
    load[machine] = load.get(machine, 0) + 1
print("Keys on each machine:", dict(sorted(load.items())))

ring.add("machine-5")
moved = sum(1 for key in keys if ring.machine_for(key) != before[key])
print(f"\nAfter adding machine-5, {moved:,} of {len(keys):,} keys change machine: {moved / len(keys):.0%}")

About a fifth of the keys move, and all of them move **to the new machine**. That is the least that could possibly move, since the new machine has to be given a fair share of something. The lookup uses binary search on a sorted list, the tool of Notebook 03.

Each machine was placed at 100 points around the circle, not one. With a single point per machine the arcs between machines would be of very unequal length, and so would the loads.

## 7. Replication, and the cost of copies

### Intuition first
With data on many machines, machines **will** fail: with a thousand of them, something breaks every day. So each piece of data is kept on several machines. These copies are **replicas**.

Copies raise a question that a single machine never faced. A value is written to one replica. For a moment, the others still hold the old value. What should a reader be told in that moment?

Let us simulate three replicas of one small store. A write goes to one replica straight away, and travels to the others a little later.

In [ ]:
class ReplicatedStore:
    def __init__(self, replicas=3):
        self.replicas = [{} for i in range(replicas)]        # each replica: key -> (version, value)
        self.in_transit = []                                 # copies on their way to other replicas
        self.version = 0

    def write(self, key, value, wait_for=1):
        """Store the value on `wait_for` replicas now. The remaining copies travel later."""
        self.version += 1
        for number in range(len(self.replicas)):
            if number < wait_for:
                self.replicas[number][key] = (self.version, value)
            else:
                self.in_transit.append((number, key, self.version, value))

    def read(self, key, ask=1, start=0):
        """Ask `ask` replicas, beginning with replica number `start`, and return the newest answer."""
        answers = []
        for i in range(ask):
            replica = self.replicas[(start + i) % len(self.replicas)]
            if key in replica:
                answers.append(replica[key])
        return max(answers)[1] if answers else None

    def deliver(self):
        """The copies in transit arrive."""
        for number, key, version, value in self.in_transit:
            if key not in self.replicas[number] or self.replicas[number][key][0] < version:
                self.replicas[number][key] = (version, value)
        self.in_transit = []

store = ReplicatedStore()
store.write("stock:Book 12", 5)
store.deliver()                                              # all three replicas now agree: 5 copies

store.write("stock:Book 12", 4)                              # one copy sold: written to replica 0 only, for now

print("Reading from replica 0:", store.read("stock:Book 12", start=0))
print("Reading from replica 2:", store.read("stock:Book 12", start=2))

store.deliver()
print("A moment later, replica 2:", store.read("stock:Book 12", start=2))

Two customers looking at the same book at the same instant are told different things, depending on which replica happens to answer. A moment later the copies have caught up and everyone agrees. This behaviour is called **eventual consistency**: if the writes stop, all replicas will *eventually* hold the same value, and until then a read may return a stale one.

### Trading speed for certainty
We can remove the stale reads by asking for more. Wait for a write to reach **W** replicas before calling it done, and ask **R** replicas on every read, taking the newest answer. If $R + W > N$, where $N$ is the number of replicas, the replicas written and the replicas read must overlap in at least one machine, so every read sees the latest write.

In [ ]:
def stale_reads(wait_for, ask, trials=3000):
    rng = random.Random(0)
    stale = 0
    for trial in range(trials):
        store = ReplicatedStore(3)
        store.write("x", "old", wait_for=3)
        store.write("x", "new", wait_for=wait_for)           # the copies to the other replicas are still travelling
        if store.read("x", ask=ask, start=rng.randrange(3)) != "new":
            stale += 1
    return stale / trials

print(f"{'W: replicas written':>20} {'R: replicas read':>18} {'R + W':>6} {'stale reads':>12}")
for wait_for, ask in [(1, 1), (1, 2), (2, 1), (2, 2), (1, 3), (3, 1)]:
    print(f"{wait_for:>20} {ask:>18} {wait_for + ask:>6} {stale_reads(wait_for, ask):>11.0%}")

Whenever $R + W$ is greater than 3, no read is ever stale. Whenever it is not, some are. But every extra replica that must be waited for makes each operation slower, and makes it **fail** if that replica is unreachable. With $W = 3$, a single broken machine stops all writing.

### Formal definition: the CAP theorem
That observation, generalised, is the best-known result about distributed data. When the network between the machines is cut, so that some replicas cannot talk to others (a **partition**), a system must choose:

- **Consistency**: refuse to answer unless it can be sure the answer is the latest, or
- **Availability**: answer with what it has, which may be stale.

It cannot do both. Partitions do happen, so every distributed store has to decide which of the two it gives up when one occurs. A bank balance calls for consistency. A count of "likes" calls for availability. The relational databases of Module A were built for a single machine, where this choice never arises, and they promise consistency always. Many NoSQL systems choose availability, and tell you so.

## 8. The real thing: MongoDB

*This section installs `mongomock`, a few hundred kilobytes.*

The query notation of our `Collection` class was MongoDB's. So the queries of Section 2 run unchanged. `mongomock` behaves like a MongoDB server that lives inside the notebook.

In [ ]:
%pip install -q mongomock
import mongomock

client = mongomock.MongoClient()
shop = client["bookshop"]                      # a database
mongo_products = shop["products"]              # a collection

mongo_products.insert_many(copy.deepcopy(catalogue))

for query in [{"kind": "book"}, {"price": {"$lt": 20}}, {"details.pages": {"$gte": 300}}, {"tags": "bestseller"}]:
    print(f"{str(query):<36} {[document['name'] for document in mongo_products.find(query)]}")

The real system offers a great deal more than our sixty lines: indexes on any field, including nested ones, and an **aggregation pipeline**, which plays the part of `GROUP BY`. A pipeline is a list of stages, each transforming the stream of documents produced by the one before.

In [ ]:
mongo_orders = shop["orders"]
mongo_orders.insert_many([{key: value for key, value in order.items() if key != "_id"} for order in orders.find()])

pipeline = [
    {"$unwind": "$lines"},                                                   # one document per order line
    {"$group": {"_id": "$lines.category",                                    # group by category
                "revenue": {"$sum": {"$multiply": ["$lines.price", "$lines.quantity"]}}}},
    {"$sort": {"revenue": -1}},
    {"$limit": 4},
]
for row in mongo_orders.aggregate(pipeline):
    print(f"{row['_id']:<10} {row['revenue']:>12,.0f}")

`$unwind` is worth noticing. To total revenue by category, each order must first be taken apart into its lines. The embedded design that made reading one order so easy makes analysis **across** orders more awkward. It is the mirror image of the relational design, where the lines were already separate rows and it was assembling one order that took a join.

## 9. Where we stand

| | Relational | Document | Key-value |
|---|---|---|---|
| A record is | a row with fixed columns | a nested document | an opaque value under a key |
| Schema | declared, enforced on every write | none enforced | none |
| Related data | separate tables, joined | embedded, or referenced by hand | none |
| Questions it answers well | almost any, including ones nobody planned | by the fields of a document | "the value for this key" only |
| Spreading over many machines | difficult | designed for it | easiest |
| Strong guarantees | yes | varies | usually few |
| Good for | most business data | varied or nested records read as a whole | sessions, caches, counters |

NoSQL did not replace SQL. After fifteen years the commonest arrangement is a relational database for the core records of a business, with a document or key-value store beside it for the particular jobs each does best. The useful skill is the one this notebook practised: for each piece of data, ask how it will be **read**, how it will **change**, and what must **never** be allowed to go wrong.

With sharding we crossed a line. For the first time in this course the data lives on **more than one machine**. So far we only stored it there. The next module asks the harder question: how do you **compute** over data that no single machine can hold?

---
## ✏️ Exercises

### Exercise 1 (Queries on documents)
Using the `products` collection, write queries for: (a) products that have at least one review field at all, (b) products costing between 15 and 130 euros inclusive, (c) products tagged `travel`, (d) products whose details mention an `author`.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
print("(a)", names(products.find({"reviews": {"$exists": True}})))
print("(b)", names(products.find({"price": {"$gte": 15, "$lte": 130}})))
print("(c)", names(products.find({"tags": "travel"})))
print("(d)", names(products.find({"details.author": {"$exists": True}})))
```

*In (b) two conditions on the same field are written inside one inner document. The products whose price is stored as text, or under a misspelt name, are silently left out of (b): a comparison with a missing value is false.*
</details>

### Exercise 2 (Add an operator)
Our `matches` function cannot yet answer *"products with at least two reviews"*. Add an operator `$size_gte` so that `{"reviews": {"$size_gte": 2}}` matches documents whose `reviews` list has at least two elements. (Add one entry to the `COMPARISONS` dictionary.)

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
COMPARISONS["$size_gte"] = lambda value, operand: isinstance(value, list) and len(value) >= operand

print(names(products.find({"reviews": {"$size_gte": 2}})))
print(names(products.find({"tags": {"$size_gte": 2}, "kind": "book"})))
```

*One line, because the function was written so that operators are data. (Real MongoDB has `$size` for an exact length. A range needs a different technique.) A query language is, in the end, a small interpreter like this one.*
</details>

### Exercise 3 (Clean up a collection)
Write a function `repair(collection)` that goes through every document and fixes the three kinds of damage from Section 3: a field called `prcie` should become `price`, a `kind` should be in lower case, and a `price` stored as text should become a number. Run it on `products` and show with `problems_with` that the documents are clean. What could such a repair **not** fix?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def repair(collection):
    for document in collection.find():
        if "prcie" in document:
            document["price"] = document.pop("prcie")
        if isinstance(document.get("kind"), str):
            document["kind"] = document["kind"].lower()
        if isinstance(document.get("price"), str):
            document["price"] = float(document["price"])

repair(products)
for document in products.find():
    found = problems_with(document)
    print(f"{document['name']:<16} {'ok' if not found else found}")
print("Under 20 euros:", names(products.find({"price": {"$lt": 20}})))
```

*The repair can fix mistakes that follow a pattern we know about. It could not tell that the reading lamp, now a "book" in lower case, is still in the wrong category, nor recover a value that was never recorded. Cleaning data afterwards is always more expensive, and less complete, than refusing bad data at the door.*
</details>

### Exercise 4 (Embedded or referenced?)
For each of the following, say whether you would embed or reference, and why. Then check your reasoning for (b) with a measurement: in the `orders` collection, how many documents would have to be changed if *Customer 42* moved to another city?

(a) The delivery address of an order. (b) The customer's current city, in each of their orders. (c) The comments on a blog post. (d) The author of a blog post, who has written 500 posts.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
affected = orders.find({"customer.id": 42})
print(len(affected), "order documents contain a copy of this customer's city")

# (a) Embed. It belongs to the order, is read with it, and must NOT change if the customer moves later.
# (b) Reference. It is shared by all the customer's orders and changes, so one copy is right.
# (c) Embed, if posts have a modest number of comments: they are read together with the post.
# (d) Reference. One author, 500 posts: embedding means 500 copies to keep in step.
```

*The questions to ask are always the same. Is it read together with its parent? Does it belong to that parent alone? Does it change, and if it does, should the old copies change with it?*
</details>

### Exercise 5 (A machine fails)
Using the `Ring` class, create a ring of five machines and record where each of the 100,000 `keys` lives. Then remove `machine-3` (delete its points from `ring.points`). What share of the keys change machine, and where do they go? Compare with the remainder method going from 5 machines to 4.

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
ring = Ring([f"machine-{i}" for i in range(1, 6)])
before = {key: ring.machine_for(key) for key in keys}

ring.points = [(position, machine) for position, machine in ring.points if machine != "machine-3"]
after = {key: ring.machine_for(key) for key in keys}

moved = [key for key in keys if after[key] != before[key]]
print(f"Consistent hashing: {len(moved) / len(keys):.0%} of the keys moved")
print("All of them were on the failed machine:", all(before[key] == "machine-3" for key in moved))

destinations = {}
for key in moved:
    destinations[after[key]] = destinations.get(after[key], 0) + 1
print("They went to:", dict(sorted(destinations.items())))

by_remainder = sum(1 for key in keys if shard_by_remainder(key, 5) != shard_by_remainder(key, 4))
print(f"\nRemainder method, 5 machines to 4: {by_remainder / len(keys):.0%} of the keys moved")
```

*Only the keys of the failed machine move, and because each machine sits at many points on the circle, they are shared out among all the survivors and do not all land on one neighbour. With remainders, the loss of one machine in five disturbs four fifths of the data.*
</details>

### Exercise 6 (How many copies to ask? A little harder)
Extend the experiment of Section 7 to **five** replicas. For every combination of W from 1 to 5 and R from 1 to 5, compute the share of stale reads, and print the results as a table. Check that the stale reads vanish exactly when $R + W > 5$. Which combination would you choose for a system that is read a thousand times for every write?

In [ ]:
# Your solution here:

<details><summary>💡 Show solution</summary>

```python
def stale_share(replicas, wait_for, ask, trials=1000):
    rng = random.Random(0)
    stale = 0
    for trial in range(trials):
        store = ReplicatedStore(replicas)
        store.write("x", "old", wait_for=replicas)
        store.write("x", "new", wait_for=wait_for)
        if store.read("x", ask=ask, start=rng.randrange(replicas)) != "new":
            stale += 1
    return stale / trials

print("        " + "".join(f"R={ask:<6}" for ask in range(1, 6)))
for wait_for in range(1, 6):
    print(f"W={wait_for}   " + "".join(f"{stale_share(5, wait_for, ask):>6.0%}  " for ask in range(1, 6)))
```

*The zeros fill exactly the region where R + W is at least 6. For a system that is read far more often than it is written, choose a large W and R = 1: writes are slow and wait for every replica, and each of the many reads asks a single machine. The price is that one unreachable replica blocks all writes. A system written far more than it is read would choose the opposite corner.*
</details>

## 🔑 Key takeaways

- A **document** is a nested, self-describing record. Documents in one collection need not share the same fields.
- Queries are themselves documents, and can reach into nested fields and lists.
- Without an enforced schema, any mistake is accepted silently. The schema moves into the code that reads the data: **schema-on-read**.
- **Embed** what is read together and belongs to one parent. **Reference** what is shared and changes. Embedding is denormalisation, with its usual price.
- A **key-value store** offers only get and put, and for that reason is the easiest kind of database to spread across machines.
- **Sharding** splits data by key. **Consistent hashing** lets machines join and leave while moving only a small share of the keys.
- **Replicas** protect against failure and can disagree for a while: **eventual consistency**. Reading and writing a majority restores certainty at the cost of speed.
- The **CAP theorem**: during a network partition, a system must give up either consistency or availability.

### 🏁 You have completed Module B: Beyond Tables!
You have seen what lies under a table, and what lies beyond one: files laid out by column for analysis, records that carry their own structure, and data spread and copied across machines.

**Next:** *Module C: Too Big for One Machine*, starting with *Notebook 07: Larger than Memory*. Before reaching for a cluster, we find out how far a single machine can go when it stops trying to hold everything at once.

---
*End of Module B.*

---
*© Ioannis Tsioulis. *From One Table to a Cluster: Working with Data at Scale*. Prepared for educational use.*